# PVT and Monte Carlo Mismatch Characterization of a 130-nm CMOS Current-Starved Ring Oscillator Using Open-Source Tools

## Análise de robustez de um CSRO-VCO em tecnologia CMOS de 130 nm

Inicialmente, este projeto apresenta o desenvolvimento e a caracterização por simulação de um Current-Starved Ring Oscillator Voltage-Controlled Oscillator (CSRO-VCO) utilizando a tecnologia CMOS de 130 nm da SkyWater e ferramentas open source. Assim, o trabalho concentra-se na análise do comportamento do oscilador frente a variações de processo, tensão de alimentação e temperatura (PVT), além dos efeitos de mismatch entre dispositivos.

Indubitavelmente, o objetivo principal é avaliar a estabilidade da frequência de oscilação e a capacidade de sintonia do circuito em relação à banda ISM de 2,4 GHz, com aplicação potencial em sistemas de comunicação sem fio, incluindo Bluetooth.

Por conta disso, o projeto estabelece como referência uma frequência nominal de 2,45 GHz, dentro da faixa de 2,400 a 2,4835 GHz da banda ISM de 2,4 GHz. As simulações investigam a influência da tensão de controle, das condições PVT e das variações estatísticas dos dispositivos sobre a frequência de oscilação.


Portanto, o projeto busca demonstrar um fluxo de desenvolvimento reproduzível para circuitos integrados analógicos de RF utilizando ferramentas open source, documentando as condições de simulação, os resultados obtidos e as limitações identificadas durante a caracterização.

## Organização e ferramentas Open Source

### Organização do notebook

| Capítulo | Tema central  |
|---|---|
| **01** | Contextualização |
| **02** | O inversor CMOS|  
| **03** | Ring Oscillator | 
| **04** | Current-Starved Ring Oscillator-Based Voltage-Controlled Oscillator | 
| **05** | PVT and Monte Carlo Mismatch | 
| **06** | Reproducibility | 
| **07** | Conclusions | 

A sequência é:

$$
\boxed{
\text{Contextualização}
\rightarrow
\text{Inversor}
\rightarrow
\text{Oscilador}
\rightarrow
\text{VCO}
\rightarrow
\text{Análise PVT e MM}
\rightarrow
\text{Reprodução}
\rightarrow
\text{Conclusão}
}
$$


Até o momento, o trabalho está concentrado na caracterização pré-layout do circuito, incluindo análise de sintonia, PVT e mismatch. A implementação física, a extração de parasitas e a validação pós-layout permanecem como etapas futuras, especialmente importantes para verificar o impacto das capacitâncias parasitas e das interconexões sobre a frequência e a robustez do oscilador.

### Ferramentas Open Source

| Ferramenta | Funcionalidade | Versão | Mantenedor |
|---|---|---|---|
| **SkyWater 130 nm PDK** | Modelos de dispositivos e regras de projeto CMOS | sky130A | SkyWater Technology / Google |
| **Python** | Automação, análise de dados e processamento dos resultados | 3.10.12 | Python Software Foundation |
| **Xschem** | Captura e edição de esquemáticos eletrônicos | 3.4.8RC | Stefan Schippers |
| **ngspice** | Simulação de circuitos analógicos e análise SPICE | 45 | Projeto ngspice |
| **Matplotlib** | Geração de gráficos e visualização dos resultados das simulações | 3.10.7 | Projeto Matplotlib |
| **Git** | Controle de versão, rastreabilidade e reprodutibilidade do projeto | 2.34.1 | Projeto Git |

## 1. Contextualização

### O que é a banda ISM?

A sigla **ISM** vem de *Industrial, Scientific and Medical* (Industrial, Científica e Médica). Ela, por sua vez, identifica faixas do espectro de radiofrequência destinadas originalmente a aplicações industriais, científicas e médicas. Algumas dessas faixas também são utilizadas por sistemas de comunicação sem fio.

Exemplos da utilização de bandas livres:

 - **Banda dos 2,4 GHz**: WiFi, Bluetooth e Zigbee.
 - **Banda dos 5 GHz**: WiFi
 - **Banda dos 868 MHz**: Zigbee, LoRa o Sigfox
 - **Banda de 434 MHz**
 - **Frequências RFID**: (13,56 MHz, 125 kHz, etc).


Inicialmente, a faixa conhecida como ISM de 2,4 GHz vai de 2,400 GHz a 2,4835 GHz em grande parte do mundo. Ela é utilizada por tecnologias como Bluetooth, Wi-Fi e Zigbee. Como vários sistemas compartilham o mesmo espectro, podem ocorrer interferências; por isso, protocolos e equipamentos empregam técnicas como divisão em canais, salto de frequência e controle de potência. Assim, a designação ISM não significa uso irrestrito: limites de potência, emissões e demais requisitos dependem das regras regulatórias de cada país.

<p align="center">
  <img src="figures/588_ISM_Bands.jpg" alt="Espectros" width="400">
</p>


### O que é um VCO?

Um **VCO** (*Voltage-Controlled Oscillator*, ou oscilador controlado por tensão) é um oscilador cuja frequência pode ser ajustada por uma tensão de controle, $Vin_{VCO}$. Sua característica pode ser representada por uma função $f(Vin_{VCO})$: ao variar a tensão, altera-se a frequência de oscilação dentro de uma faixa de sintonia.

<p align="center">
  <img src="figures/vco.jpg" alt="Oscilador controlado por tensão" width="400">
</p>

Assim, a ideia de todo o projeto é fornecer um VCO que opere encobrindo a banda ISM.

## 2. O inversor CMOS

### O que é um inversor CMOS?

O **inversor CMOS** é uma porta lógica de um bit que produz na saída o complemento do sinal aplicado à entrada: quando a entrada está em nível baixo, a saída fica alta; quando a entrada está alta, a saída fica baixa. CMOS significa *Complementary Metal-Oxide-Semiconductor* e descreve o uso complementar de transistores PMOS e NMOS.

<p align="center">
  <img src="figures/inv1.jpg" alt="inv" width="200">
</p>

### Estrutura e funcionamento

O inversor CMOS básico possui dois transistores: um PMOS ligado entre a alimentação positiva ($V_{DD}$) e a saída, e um NMOS ligado entre a saída e o terra (GND). As portas (*gates*) dos dois transistores são conectadas à entrada. O PMOS funciona como caminho de subida da saída, enquanto o NMOS funciona como caminho de descida.

<p align="center">
  <img src="figures/inversor.jpg" alt="inv" width="200">
</p>


| Entrada | PMOS | NMOS | Saída |
|---|---|---|---|
| Baixa | Ligado | Desligado | Alta |
| Alta  | Desligado | Ligado | Baixa |

Quando a entrada está baixa, o PMOS conduz e carrega a capacitância da saída em direção a $V_{DD}$; o NMOS permanece desligado. Quando a entrada está alta, o NMOS conduz e descarrega essa capacitância em direção ao terra; o PMOS fica desligado. Durante a transição entre os níveis, ambos podem conduzir por um curto intervalo, criando um caminho momentâneo de corrente entre alimentação e terra.

<p align="center">
  <img src="figures/charge.jpg" alt="cg" width="200">
</p>

<p align="center">
  <img src="figures/discharge.jpg" alt="inv" width="200">
</p>

### Figuras de Mérito essenciais

O **atraso de propagação** é o tempo entre uma transição na entrada do inversor e a transição correspondente na saída. Como as transições de subida e descida percorrem caminhos diferentes — a subida é conduzida principalmente pelo PMOS e a descida pelo NMOS — seus atrasos são medidos separadamente.

- **$t_{PLH}$** (*Propagation Low-to-High*): atraso associado à saída que muda de nível baixo para alto. No inversor, essa transição ocorre quando a entrada muda de alto para baixo. Mede-se o intervalo entre o cruzamento de 50% de $V_{DD}$ pela entrada na borda de descida e o cruzamento de 50% de $V_{DD}$ pela saída na borda de subida.
- **$t_{PHL}$** (*Propagation High-to-Low*): atraso associado à saída que muda de nível alto para baixo. No inversor, isso ocorre quando a entrada muda de baixo para alto. Mede-se o intervalo entre o cruzamento de 50% de $V_{DD}$ pela entrada na borda de subida e o cruzamento de 50% de $V_{DD}$ pela saída na borda de descida.

<p align="center">
  <img src="figures/images.jpeg" alt="inv" width="400">
</p>

$$
t_p = \frac{t_{PLH} + t_{PHL}}{2}
$$

Por fim, é necessário ressaltar a importância desta equação, pois ela será fundamental pra descoberta da frequência de operação do nosso ring oscillator.


Para um inversor CMOS, os atrasos também podem ser estimados a partir dos parâmetros elétricos dos transistores e da capacitância de carga. Considerando o modelo de primeira ordem:

$$
t_{PHL} = t_{HL} = R_n C_{ox} = \tau_n
$$

$$
t_{PLH} = t_{LH} = R_p C_{ox} = \tau_p = 3\tau_n
$$

em que $R_n$ e $R_p$ representam as resistências equivalentes dos transistores NMOS e PMOS, respectivamente, e $C_{ox}$ representa a capacitância de carga considerada no modelo. A relação $\tau_p = 3\tau_n$ decorre da hipótese adotada para as resistências equivalentes dos transistores.

Substituindo essas relações na expressão do atraso médio de propagação, obtém-se:

$$
t_p = \frac{t_{PLH}+t_{PHL}}{2}
=\frac{\tau_p+\tau_n}{2}
=\frac{3\tau_n+\tau_n}{2}
=2\tau_n
$$


## 3. Ring Oscillator

### O que é um Oscilador em Anel?


Um **oscilador em anel** (*ring oscillator*) é um circuito realimentado formado por um número ímpar de estágios inversores. Nesta seção, considera-se uma cadeia de **três inversores CMOS**: a saída do terceiro estágio é conectada à entrada do primeiro. Em uma representação de bloco, podem ser mostrados apenas o sinal de entrada da cadeia ($v_{in}$) e o sinal de saída após o terceiro estágio ($v_{out}$); os nós intermediários existem fisicamente, mas não precisam ser expostos como terminais de medição.

<p align="center">
  <img src="figures/ro.jpg" alt="inv" width="400">
</p>


### Critérios de Barkhausen

Os Critérios de Barkhausen estabelecem duas condições para a oscilação em um sistema realimentado
$$
|A(j\omega_0)\beta(j\omega_0)|\geq 1
$$

$$
\arg{A(j\omega_0)\beta(j\omega_0)}=2\pi k,\quad k\in\mathbb{Z}.
$$


O primeiro critério exige ganho de laço igual ou superior à unidade para iniciar a oscilação, enquanto o segundo exige fase total equivalente a $360^\circ k$, garantindo realimentação construtiva. Em regime permanente, o ganho de laço tende à unidade.

<p align="center">
  <img src="figures/image.png" alt="dw" width="300">
</p>

Em um ring oscillator com número ímpar de inversores, a inversão total em baixas frequências é de $180^\circ$. Assim, os atrasos de propagação introduzem a defasagem adicional necessária para satisfazer o critério de fase e sustentar a oscilação. Então, esses critérios são necessários, mas não suficientes para garantir a oscilação.


### Estimativa do período e da frequência

Para três estágios, uma estimativa de primeira ordem é:

$$
T_{osc} \approx \sum_{i=1}^{3}(t_{PHL,i}+t_{PLH,i}).
$$

Se os três estágios forem aproximadamente iguais, isto é, $t_{PHL,i}\approx t_{PHL}$ e $t_{PLH,i}\approx t_{PLH}$, então:

$$
T_{osc} \approx 3(t_{PHL}+t_{PLH}).
$$

Usando o atraso médio de um estágio, $t_p=(t_{PHL}+t_{PLH})/2$, obtém-se:

$$
T_{osc}\approx 6t_p,
\qquad
f_{osc}\approx \frac{1}{T_{osc}}
=\frac{1}{3(t_{PHL}+t_{PLH})}
=\frac{1}{6t_p}.
$$




## 4. Current Starved Ring Oscillator - VCO

### Sintonia por resistência variável

A frequência de um oscilador em anel depende do atraso dos inversores. Uma forma de ajustá-la é controlar a resistência no caminho de descarga: ao reduzir $V_{cont}$, os transistores NMOS de controle conduzem menos, a saída demora mais para cair e a frequência de oscilação diminui.

<p align="center">
  <img src="figures/csro.png" alt="inv" width="500"
</p>

Essa técnica pode tornar as transições de descida muito mais lentas que as de subida, aumentando a assimetria e o ruído de fase. Além disso, perto da tensão de limiar dos NMOS, a resistência cresce rapidamente, deixando a sintonia não linear; em valores muito baixos de $V_{cont}$, o circuito pode parar de oscilar. Por fim, um caminho resistivo fixo em paralelo limita esse aumento e mantém uma frequência mínima, mas não elimina a assimetria e também reduz a frequência máxima.

Por vez, isso também pode ser feito pro ramo de pull
up de modo que a topologia fique desta forma:

<p align="center">
  <img src="figures/csro-2.jpg" alt="inv" width="400"
</p>

### Mensuração teórica da corrente de polarização

Antes do dimensionamento dos transistores limitadores de corrente, estima-se a corrente necessária para atingir as frequências desejadas na banda ISM de 2,4 GHz.

<p align="center">
  <img src="figures/modelo.jpg" alt="Modelo do oscilador em anel com limitadores de corrente" width="200">
</p>

Considerando um oscilador em anel CMOS de três estágios, a frequência de oscilação pode ser aproximada por:

$$
f_0 = \frac{I}{NCV_{DD}}
$$

Isolando a corrente:

$$
I = f_0NCV_{DD}
$$

Adotam-se os seguintes parâmetros:

- Capacitância por estágio: $C = 3\,\text{fF}$;
- Número de estágios: $N = 3$;
- Tensão de alimentação: $V_{DD} = 1{,}8\,\text{V}$.

O atraso médio por estágio é estimado por:

$$
t_{\text{center}} = \frac{1}{2Nf_0}
$$

#### Frequência de 2,400 GHz

A corrente necessária é:

$$
\begin{aligned}
I &= f_0NCV_{DD} \\
  &= (2{,}4\times10^9)(3)(3\times10^{-15})(1{,}8) \\
  &= 38{,}88\times10^{-6}\,\text{A}
\end{aligned}
$$

$$
\boxed{I = 38{,}88\,\mu\text{A}}
$$

O atraso médio por estágio é:

$$
\begin{aligned}
t_{\text{center}} &= \frac{1}{2Nf_0} \\
&= \frac{1}{2(3)(2{,}4\times10^9)}
\end{aligned}
$$

$$
\boxed{t_{\text{center}} \approx 69{,}44\,\text{ps}}
$$

#### Frequência central de 2,450 GHz

A corrente necessária é:

$$
\begin{aligned}
I &= f_0NCV_{DD} \\
  &= (2{,}45\times10^9)(3)(3\times10^{-15})(1{,}8) \\
  &= 39{,}69\times10^{-6}\,\text{A}
\end{aligned}
$$

$$
\boxed{I = 39{,}69\,\mu\text{A}}
$$

O atraso médio por estágio é:

$$
\begin{aligned}
t_{\text{center}} &= \frac{1}{2Nf_0} \\
&= \frac{1}{2(3)(2{,}45\times10^9)}
\end{aligned}
$$

$$
\boxed{t_{\text{center}} \approx 68{,}03\,\text{ps}}
$$

#### Frequência de 2,4835 GHz

A corrente necessária é:

$$
\begin{aligned}
I &= f_0NCV_{DD} \\
  &= (2{,}4835\times10^9)(3)(3\times10^{-15})(1{,}8) \\
  &= 40{,}2327\times10^{-6}\,\text{A}
\end{aligned}
$$

$$
\boxed{I \approx 40{,}23\,\mu\text{A}}
$$

O atraso médio por estágio é:

$$
\begin{aligned}
t_{\text{center}} &= \frac{1}{2Nf_0} \\
&= \frac{1}{2(3)(2{,}4835\times10^9)}
\end{aligned}
$$

$$
\boxed{t_{\text{center}} \approx 67{,}11\,\text{ps}}
$$

#### Resumo dos resultados

| Frequência (GHz) | Corrente ($\mu$ A) | Atraso médio por estágio (ps) |
|:---:|---:|---:|
| 2,4000 | 38,88 | 69,44 |
| 2,4500 | 39,69 | 68,03 |
| 2,4835 | 40,23 | 67,11 |

Os resultados fornecem uma estimativa inicial da corrente necessária para cobrir a faixa de 2,4000 a 2,4835 GHz. O dimensionamento definitivo dos transistores limitadores deve ser validado por simulações, considerando as capacitâncias parasitas e as variações de processo, tensão e temperatura (PVT).

### Controle de entrada

A configuração de controle utilizada até o momento apresenta uma limitação: são necessárias duas entradas independentes para controlar a frequência de oscilação, dificultando a implementação de um ajuste unificado.

Para solucionar esse problema, propõe-se a utilização de um espelho de corrente, responsável por estabelecer uma corrente de polarização controlada e distribuí-la aos estágios do ring oscillator. Dessa forma, cada estágio recebe uma corrente definida, permitindo controlar a frequência de oscilação por meio de uma única entrada de referência, simplificando a interface de controle e garantindo maior uniformidade na polarização dos estágios.

Assim, o circuito final ficará desta forma:

<p align="center">
  <img src="figures/csro-f.jpg" alt="Modelo do oscilador em anel com limitadores de corrente" width="500">
</p>

> 🛈 Nota: Os parâmetros dimensionais e valores de capacitâncias apresentados são preliminares e foram adotados arbitrariamente para fins ilustrativos. Sua otimização deverá ser realizada posteriormente por meio de simulações.

### Simulação e caracterização do VCO

Com base na estimativa teórica de corrente apresentada anteriormente, foram definidos os parâmetros dimensionais iniciais dos transistores responsáveis pela polarização e limitação de corrente do oscilador.

A partir dessa configuração, foram realizadas simulações para caracterizar o comportamento do VCO e avaliar sua capacidade de operação na banda ISM de 2,4 GHz. A análise foi dividida em três etapas:

1. **Análise temporal da saída:** verificação da forma de onda gerada pelo oscilador, permitindo avaliar a estabilidade da oscilação e o comportamento do sinal de saída.

2. **Característica de sintonia:** análise da frequência de oscilação em função da tensão de controle ($V_{\text{cont}}$), permitindo identificar a faixa de sintonia e verificar se o VCO contempla a banda de frequência desejada.

3. **Sensibilidade de sintonia ($K_{\text{VCO}}$):** avaliação da variação da frequência em relação à tensão de controle, expressa por:

$$
K_{\text{VCO}}(V_{\text{cont}}) =
\frac{df_{\text{osc}}}{dV_{\text{cont}}}
$$

Essa grandeza representa o ganho de sintonia do oscilador, geralmente expresso em $\text{Hz/V}$ ou $\text{MHz/V}$, e permite avaliar a sensibilidade da frequência às variações da tensão de controle.

#### Resposta temporal do VCO

<p align="center">
    <img src="figures/grafico_tensao_vs_tempo.png"
         alt="Forma de onda temporal da saída do VCO"
         width="700">
</p>

#### Característica de sintonia: frequência em função da tensão de controle

<p align="center">
    <img src="figures/grafico_freq_vs_vcont.png"
         alt="Frequência de oscilação em função da tensão de controle"
         width="700">
</p>

#### Ganho de sintonia: $K_{\text{VCO}}$ em função da tensão de controle

<p align="center">
    <img src="figures/grafico_kvco_vs_vcont.png"
         alt="Ganho de sintonia do VCO em função da tensão de controle"
         width="700">
</p>

A partir dessas curvas, é possível avaliar a faixa efetiva de sintonia, a linearidade da característica $f_{\text{osc}}(V_{\text{cont}})$ e a variação do ganho de sintonia ao longo da faixa de controle. Portanto, esses resultados permitem verificar o atendimento aos requisitos de frequência estabelecidos para o projeto.


## 5. PVT and Mismatch Characterization

### O que é PVT?

PVT é a sigla para *Process, Voltage and Temperature*, representando as três principais fontes de variação que afetam o comportamento de circuitos integrados:

- **Process (Processo):** variações decorrentes das etapas de fabricação, que afetam parâmetros dos transistores, como tensão de limiar ($V_{TH}$), mobilidade dos portadores e dimensões efetivas.
- **Voltage (Tensão):** variações na tensão de alimentação ($V_{DD}$), que modificam as condições de polarização e o desempenho dos dispositivos.
- **Temperature (Temperatura):** alterações na temperatura de operação, que influenciam a mobilidade dos portadores, a tensão de limiar e as correntes dos transistores.

Essas variações podem modificar parâmetros importantes do circuito, como atrasos de propagação, consumo de potência e frequência de oscilação.

### O que é Monte Carlo Mismatch?

A análise de *Monte Carlo Mismatch* utiliza simulações estatísticas para avaliar os efeitos das variações aleatórias entre dispositivos fabricados no mesmo circuito integrado.

Diferentemente das variações globais de processo, o *mismatch* representa diferenças locais entre transistores, mesmo quando submetidos às mesmas condições de fabricação. Essas diferenças podem ocorrer, por exemplo, na tensão de limiar ($V_{TH}$), na corrente de saturação e nas dimensões efetivas dos dispositivos.

Em cada repetição da simulação, são sorteadas variações aleatórias nos parâmetros dos transistores, permitindo avaliar a dispersão estatística do desempenho do circuito.

### Por que essa análise é importante para o CSRO-VCO?

No CSRO-VCO (*Current-Starved Ring Oscillator Voltage-Controlled Oscillator*), a frequência de oscilação depende diretamente das correntes de polarização e dos atrasos de propagação dos estágios do oscilador.

Consequentemente, as variações de processo, tensão e temperatura, juntamente com o *mismatch* entre os transistores, podem provocar deslocamentos significativos na faixa de frequência de operação.

Esse comportamento é particularmente crítico para o projeto, pois o VCO deve cobrir integralmente a banda ISM de 2,4 GHz, compreendida entre 2,4000 e 2,4835 GHz. Caso as variações provoquem um deslocamento excessivo da faixa de sintonia, o oscilador poderá deixar de atender aos limites de frequência estabelecidos, mesmo que apresente desempenho adequado nas condições nominais.

### PVT com Mismatch: configuração das simulações

Para esta etapa, foram consideradas exclusivamente as condições de processo TT (*Typical-Typical*). Essa escolha parte da premissa de que as variações globais de processo podem ser compensadas por meio de calibração ou *trimming* dos dispositivos.

Foram realizadas **200 repetições de Monte Carlo Mismatch** para cada combinação de tensão de alimentação e temperatura, mantendo o processo TT.

Os parâmetros utilizados foram:

- **Processo:** TT (*Typical-Typical*);
- **Repetições de Monte Carlo:** 200 por condição de operação;
- **Tensão de alimentação:** de $1{,}71\,\text{V}$ a $1{,}89\,\text{V}$, com incrementos de $0{,}01\,\text{V}$;
- **Temperatura:** de -25 °C a 85 °C, com incrementos de 5 °C.

Dessa forma, foram avaliadas diferentes combinações de tensão e temperatura, permitindo observar tanto o deslocamento da faixa de sintonia quanto a dispersão de frequência causada pelo *mismatch* entre os dispositivos.

<p align="center">
    <img src="figures/histogram.png"
         alt="histograma"
         width="900">
</p>

O objetivo é identificar as condições mais críticas de operação e verificar se a faixa de sintonia do CSRO-VCO permanece dentro dos limites necessários para cobrir a banda ISM de 2,4 GHz.

<p align="center">
    <img src="figures/bluetooth_metrics.png"
         alt=""
         width="700">
</p>

## 6. Reprodutibilidade

A reprodução dos resultados apresentados neste trabalho requer a instalação e configuração prévia das ferramentas utilizadas no fluxo de projeto, simulação e pós-processamento do circuito.

As seguintes dependências são necessárias:

- **SKY130 PDK:** conjunto de modelos e regras de projeto da tecnologia CMOS de 130 nm, utilizado para caracterização dos dispositivos e simulação do circuito.
- **Xschem:** ferramenta de captura esquemática utilizada para descrição e elaboração dos circuitos.
- **Ngspice:** simulador SPICE responsável pela análise elétrica e pelas simulações do circuito, incluindo as caracterizações PVT e Monte Carlo Mismatch.
- **Python:** ambiente utilizado no pós-processamento dos dados, extração de métricas e geração dos gráficos para análise dos resultados.

Além da instalação dessas ferramentas, é necessário garantir a compatibilidade entre suas versões, a configuração adequada dos caminhos do PDK e a disponibilidade dos modelos de dispositivos utilizados nas simulações.

Com essas dependências devidamente configuradas, torna-se possível reproduzir o fluxo de trabalho, executar novamente as simulações e comparar os resultados obtidos com os apresentados neste estudo.

### Sinal de saída

Para simular o funcionamento do VCO para analisar a curva de saída da tensão e colocar os dados em .csv:

In [ ]:
from pathlib import Path
import subprocess
import sys

project_dir = Path.cwd()

scripts = [
    project_dir / "reproducibility" / "sinal" / "sinal.py",
    project_dir / "reproducibility" / "sint" / "sintonia.py",
]

for script in scripts:
    if not script.is_file():
        raise FileNotFoundError(f"Script não encontrado: {script}")

    print(f"\nExecutando {script.name}...")
    subprocess.run([sys.executable, str(script)], check=True)

print("\nOs dois scripts terminaram com sucesso.")


### Simulação de Curva de Sintonização e ganho do VCO

Para simular o funcionamento dos dispositivo (com PVT podendo ser variado no netlist), execute o seguinte código:

In [ ]:
from pathlib import Path
import subprocess
import sys

# Procura a raiz do projeto a partir do diretório de execução do notebook.
project_dir = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "reproducibility" / "sint" / "sintonia.py").is_file()
    ),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Não foi possível localizar a pasta reproducibility/sint.")

sint_dir = project_dir / "reproducibility" / "sint"



print("Executando a simulação de sintonia...")
subprocess.run(
    [sys.executable, str(sint_dir / "sintonia.py")],
    cwd=sint_dir,
    check=True,
)

print("\nGerando os gráficos de sintonia e Kvco...")
subprocess.run(
    [sys.executable, str(sint_dir / "sintoniapos.py")],
    cwd=sint_dir,
    check=True,
)

print("\nFluxo de sintonia concluído.")

### Simulação de PVT e mismatch

In [ ]:
from pathlib import Path
import subprocess
import sys

pasta_scripts = Path.cwd() / "reproducibility" / "pvtmm"

copynet = pasta_scripts / "copynet.py"
pipeline = pasta_scripts / "rodar_pipeline.py"

for script in (copynet, pipeline):
    if not script.is_file():
        raise FileNotFoundError(f"Script não encontrado: {script}")

    print(f"\nExecutando {script.name}...")
    subprocess.run([sys.executable, str(script)], cwd=pasta_scripts, check=True)

print("\nExecução concluída com sucesso.")

> 🛈 Atenção: as simulações são pesadas e podem demorar de minutos a dias. Caso não consiga simular mesmo assim, execute manualmente os aquivos .py pelo terminal dentro do diretório do código executado.

> Caso precise alterar o circuito abra o .sch do arquivo pelo Xschem e coloque as alterações que desejas, ao clicar em gerar netlist (configure um netlist spice) ele irá gerar um netlist do formato .spice que poderá ser executado pelos códigos python deste notebook.

## 7. Conclusão

A caracterização realizada permitiu avaliar o comportamento do CSRO-VCO sob diferentes condições de operação, considerando as variações de processo, tensão de alimentação, temperatura e *mismatch* entre dispositivos.

Os resultados das simulações indicam que o circuito apresenta comportamento reprodutível na maior parte das condições avaliadas, desde que respeitados os limites de operação estabelecidos. Entretanto, observou-se que as variações de tensão e temperatura, associadas ao *mismatch*, podem provocar deslocamentos significativos na faixa de frequência de oscilação.

Esse comportamento evidencia que o dimensionamento nominal dos transistores e a definição da corrente de polarização, isoladamente, não são suficientes para garantir a cobertura da banda ISM de 2,4 GHz em todas as condições de operação.

Portanto, a robustez do projeto depende da adequada definição da corrente de polarização, da margem de sintonia disponível e da implementação de mecanismos de calibração (*trimming*), capazes de compensar os desvios de frequência decorrentes das variações dos dispositivos.

Conclui-se que a análise PVT e Monte Carlo Mismatch é indispensável para avaliar a viabilidade do CSRO-VCO, identificar as condições críticas de operação e estabelecer os requisitos necessários para uma implementação robusta em tecnologia CMOS de 130 nm.

## 8. Referências

RAZAVI, Behzad. **Design of Analog CMOS Integrated Circuits**. 2. ed. New York: McGraw-Hill, 2017.

RAZAVI, Behzad. **Design of CMOS Phase-Locked Loops: From Circuit Level to Architecture Level**. Cambridge: Cambridge University Press, 2020.

ALEXANDER, Charles K.; SADIKU, Matthew N. O. **Fundamentos de circuitos elétricos**. 5. ed. Porto Alegre: AMGH, 2013.

LAGRIDA. **Online LaTeX Equation Editor**. 2025. Disponível em: <https://latexeditor.lagrida.com/>. Acesso em: 30 out. 2026.

SEDRA, Adel S.; SMITH, Kenneth C. **Microelectronic Circuits**. 7. ed. New York: Oxford University Press, 2015.

STEER, Michael. **Microwave and RF Design: Networks**. 3. ed. Raleigh: North Carolina State University, 2019.

WIEDEMANN, Philipp; KIRZINGER, Manuel; PFANNENMÜLLER, Christof. **CircuitikZ Designer**. 2025. Disponível em: <https://circuit2tikz.tf.fau.de/designer/>. Acesso em: 30 out. 2026.

OPENAI. **ChatGPT**. 2026. Ferramenta utilizada como auxílio na revisão e no aprimoramento da redação. Disponível em: <https://chatgpt.com/>. Acesso em: 30 out. 2026.